# Business Impact Analysis - Calculadora Interativa

Este notebook permite explorar e calcular metricas BIA de forma interativa.

**Metricas abordadas:** RPO, RTO, WRT, MTD, MOR, MTBF, MTTR

---

## 1. Setup inicial

In [ ]:
# Nao precisa de instalar nada extra - usa apenas Python standard
from datetime import datetime, timedelta
import json

## 2. Validador de metricas BIA

Uma das regras fundamentais da BIA e:

```
RTO + WRT <= MTD
```

Vamos criar uma funcao que valida esta regra e calcula as margens.

In [ ]:
def validar_bia(nome, rpo_min, rto_min, wrt_min, mtd_min):
    """
    Valida as metricas BIA de um sistema.
    Todos os valores em minutos.
    """
    soma = rto_min + wrt_min
    margem = mtd_min - soma
    valido = soma <= mtd_min
    
    print(f"{'='*60}")
    print(f" BIA - {nome}")
    print(f"{'='*60}")
    print(f"  RPO: {rpo_min} min ({rpo_min/60:.1f}h)")
    print(f"  RTO: {rto_min} min ({rto_min/60:.1f}h)")
    print(f"  WRT: {wrt_min} min ({wrt_min/60:.1f}h)")
    print(f"  MTD: {mtd_min} min ({mtd_min/60:.1f}h)")
    print(f"{'─'*60}")
    print(f"  RTO + WRT = {soma} min ({soma/60:.1f}h)")
    print(f"  MTD       = {mtd_min} min ({mtd_min/60:.1f}h)")
    print(f"  Margem    = {margem} min ({margem/60:.1f}h)")
    print(f"{'─'*60}")
    
    if valido:
        print(f"  RESULTADO: VALIDO (RTO + WRT <= MTD)")
        if margem < mtd_min * 0.1:
            print(f"  AVISO: Margem muito apertada ({margem/mtd_min*100:.1f}% do MTD)")
    else:
        print(f"  RESULTADO: INVALIDO! RTO + WRT excede MTD em {abs(margem)} min")
        print(f"  ACAO: Reduzir RTO em {abs(margem)} min, ou reduzir WRT, ou aumentar MTD")
    
    print(f"{'='*60}")
    print()
    return valido

### Testar com os exemplos da aula

In [ ]:
# Exemplo 1 - Loja online
validar_bia("Loja Online (E-commerce)", rpo_min=15, rto_min=60, wrt_min=120, mtd_min=240)

# Exemplo 2 - Hospital
validar_bia("Hospital (Urgencia)", rpo_min=1, rto_min=15, wrt_min=30, mtd_min=60)

# Exemplo 3 - Universidade
validar_bia("Universidade (Matriculas)", rpo_min=60, rto_min=120, wrt_min=240, mtd_min=480)

### Testar com um caso invalido

O que acontece quando RTO + WRT > MTD?

In [ ]:
# Caso invalido - empresa de contabilidade (Exercicio 2.1)
validar_bia("Contabilidade (INVALIDO)", rpo_min=120, rto_min=300, wrt_min=240, mtd_min=480)

### Experimenta com os teus proprios valores

In [ ]:
# Altera os valores abaixo e executa a celula
validar_bia(
    nome="O meu sistema",
    rpo_min=60,     # em minutos
    rto_min=120,    # em minutos
    wrt_min=180,    # em minutos
    mtd_min=360     # em minutos
)

---

## 3. Calculadora MTBF / MTTR / Disponibilidade

Calcula metricas de fiabilidade a partir de dados de falhas.

In [ ]:
def calcular_fiabilidade(nome, horas_operacao, falhas):
    """
    Calcula MTBF, MTTR e disponibilidade.
    
    Args:
        nome: Nome do sistema
        horas_operacao: Total de horas do periodo analisado
        falhas: Lista de duracoes de reparacao em horas
    """
    num_falhas = len(falhas)
    if num_falhas == 0:
        print(f"Sem falhas registadas para {nome}")
        return
    
    tempo_reparacao_total = sum(falhas)
    tempo_operacao_efetivo = horas_operacao - tempo_reparacao_total
    
    mtbf = tempo_operacao_efetivo / num_falhas
    mttr = tempo_reparacao_total / num_falhas
    disponibilidade = mtbf / (mtbf + mttr)
    
    # Downtime por ano (se 24/7)
    downtime_anual = 8760 * (1 - disponibilidade)
    
    print(f"{'='*60}")
    print(f" Fiabilidade - {nome}")
    print(f"{'='*60}")
    print(f"  Periodo analisado: {horas_operacao} horas")
    print(f"  Numero de falhas:  {num_falhas}")
    print(f"  Reparacoes:        {falhas}")
    print(f"{'─'*60}")
    print(f"  MTBF: {mtbf:.1f} horas ({mtbf/24:.1f} dias)")
    print(f"  MTTR: {mttr:.2f} horas ({mttr*60:.0f} min)")
    print(f"  Disponibilidade: {disponibilidade*100:.3f}%")
    print(f"{'─'*60}")
    print(f"  Downtime anual estimado: {downtime_anual:.1f} horas")
    print()
    
    # Comparar com SLAs comuns
    slas = {
        "99.0%":   87.6,
        "99.9%":   8.76,
        "99.95%":  4.38,
        "99.99%":  0.876,
    }
    
    print(f"  Comparacao com SLAs:")
    for sla, max_down in slas.items():
        status = "CUMPRE" if downtime_anual <= max_down else "NAO CUMPRE"
        print(f"    SLA {sla} (max {max_down:>6.2f}h/ano): {status}")
    
    print(f"{'='*60}")
    print()
    
    return {
        "mtbf": mtbf,
        "mttr": mttr,
        "disponibilidade": disponibilidade
    }

In [ ]:
# Exemplo: Empresa de hosting (Exercicio 3.1)
calcular_fiabilidade(
    nome="Cluster de Hosting",
    horas_operacao=8760,  # 1 ano, 24/7
    falhas=[1.5, 0.75, 3.25, 2.0, 1.0, 4.5]  # horas de cada reparacao
)

In [ ]:
# Exemplo: Servidor da clinica veterinaria (Exercicio 2.2)
calcular_fiabilidade(
    nome="Servidor Clinica Veterinaria",
    horas_operacao=2500,  # 10h/dia, 250 dias/ano
    falhas=[2.0, 2.0, 2.0]  # 3 falhas de 2 horas cada
)

### Experimenta com os teus dados

In [ ]:
# Altera os valores abaixo
calcular_fiabilidade(
    nome="O meu servidor",
    horas_operacao=8760,          # horas totais do periodo
    falhas=[1.0, 2.5, 0.5, 3.0]  # duracao de cada reparacao (horas)
)

---

## 4. Calculadora de custo de downtime

Estima as perdas financeiras de um incidente.

In [ ]:
def calcular_custo_downtime(nome, faturacao_hora, rto_min, wrt_min, 
                             produtividade_wrt=0.5, incidentes_ano=1):
    """
    Calcula o custo estimado de downtime.
    
    Args:
        nome: Nome do cenario
        faturacao_hora: Faturacao media por hora (EUR)
        rto_min: RTO em minutos
        wrt_min: WRT em minutos
        produtividade_wrt: % de produtividade durante WRT (0 a 1)
        incidentes_ano: Numero medio de incidentes por ano
    """
    rto_h = rto_min / 60
    wrt_h = wrt_min / 60
    
    # Durante RTO: perda total
    perda_rto = rto_h * faturacao_hora
    
    # Durante WRT: perda parcial
    perda_wrt = wrt_h * faturacao_hora * (1 - produtividade_wrt)
    
    perda_incidente = perda_rto + perda_wrt
    perda_anual = perda_incidente * incidentes_ano
    
    print(f"{'='*60}")
    print(f" Custo de Downtime - {nome}")
    print(f"{'='*60}")
    print(f"  Faturacao/hora:          {faturacao_hora:>10,.0f} EUR")
    print(f"  RTO:                     {rto_min:>10} min ({rto_h:.1f}h)")
    print(f"  WRT:                     {wrt_min:>10} min ({wrt_h:.1f}h)")
    print(f"  Produtividade no WRT:    {produtividade_wrt*100:>9.0f}%")
    print(f"  Incidentes/ano:          {incidentes_ano:>10}")
    print(f"{'─'*60}")
    print(f"  Perda durante RTO:       {perda_rto:>10,.0f} EUR")
    print(f"  Perda durante WRT:       {perda_wrt:>10,.0f} EUR")
    print(f"  Perda por incidente:     {perda_incidente:>10,.0f} EUR")
    print(f"  Perda anual estimada:    {perda_anual:>10,.0f} EUR")
    print(f"{'='*60}")
    print()
    
    return perda_anual

In [ ]:
# Exemplo: E-commerce (Exercicio 3.2)
print("CENARIO ATUAL:")
perda_atual = calcular_custo_downtime(
    "E-commerce (atual)",
    faturacao_hora=50000,
    rto_min=240,    # 4 horas
    wrt_min=120,    # 2 horas
    incidentes_ano=3
)

print("CENARIO COM INVESTIMENTO:")
perda_nova = calcular_custo_downtime(
    "E-commerce (com investimento)",
    faturacao_hora=50000,
    rto_min=60,     # 1 hora
    wrt_min=120,    # 2 horas
    incidentes_ano=3
)

investimento = 120000
poupanca = perda_atual - perda_nova
roi = poupanca - investimento

print(f"{'='*60}")
print(f" ANALISE DE INVESTIMENTO")
print(f"{'='*60}")
print(f"  Poupanca anual:          {poupanca:>10,.0f} EUR")
print(f"  Custo do investimento:   {investimento:>10,.0f} EUR")
print(f"  Beneficio liquido:       {roi:>10,.0f} EUR")
print(f"  Investimento justifica-se: {'SIM' if roi > 0 else 'NAO'}")
print(f"{'='*60}")

---

## 5. Simulador de timeline de incidente

Simula a timeline de um incidente e verifica se as metricas sao cumpridas.

In [ ]:
def simular_incidente(nome, hora_falha, rpo_min, rto_min, wrt_min, mtd_min,
                       hora_detecao=None, hora_restauro=None, hora_normal=None):
    """
    Simula a timeline de um incidente e verifica conformidade.
    
    Args:
        hora_falha: Hora da falha (formato "HH:MM")
        hora_detecao: Hora em que a falha foi detetada (opcional)
        hora_restauro: Hora em que o sistema foi restaurado (opcional)
        hora_normal: Hora em que a operacao normalizou (opcional)
    """
    falha = datetime.strptime(hora_falha, "%H:%M")
    
    # Calcular deadlines
    deadline_rto = falha + timedelta(minutes=rto_min)
    deadline_mtd = falha + timedelta(minutes=mtd_min)
    ponto_rpo = falha - timedelta(minutes=rpo_min)
    
    print(f"{'='*70}")
    print(f" Simulacao de Incidente - {nome}")
    print(f"{'='*70}")
    print()
    print(f"  TIMELINE PLANEADA:")
    print(f"  {'─'*50}")
    print(f"  RPO (dados desde):    {ponto_rpo.strftime('%H:%M')}")
    print(f"  Falha:                {falha.strftime('%H:%M')}")
    print(f"  Deadline RTO:         {deadline_rto.strftime('%H:%M')} (sistema deve estar up)")
    print(f"  Deadline MTD:         {deadline_mtd.strftime('%H:%M')} (limite maximo)")
    print()
    
    # Timeline visual
    print(f"  DIAGRAMA:")
    print(f"  {'─'*50}")
    print(f"  {ponto_rpo.strftime('%H:%M')}     {falha.strftime('%H:%M')}          {deadline_rto.strftime('%H:%M')}                  {deadline_mtd.strftime('%H:%M')}")
    print(f"    |        |             |                     |")
    print(f"    RPO      FALHA         RTO                   MTD")
    print(f"    |<-dados->|<---recup--->|<------WRT---------->|")
    print()
    
    # Se foram fornecidos tempos reais, verificar conformidade
    if hora_restauro:
        restauro = datetime.strptime(hora_restauro, "%H:%M")
        tempo_real_rto = (restauro - falha).seconds / 60
        
        print(f"  RESULTADO REAL:")
        print(f"  {'─'*50}")
        
        if hora_detecao:
            detecao = datetime.strptime(hora_detecao, "%H:%M")
            tempo_detecao = (detecao - falha).seconds / 60
            print(f"  Detecao:   {hora_detecao} ({tempo_detecao:.0f} min apos falha)")
        
        rto_ok = tempo_real_rto <= rto_min
        print(f"  Restauro:  {hora_restauro} ({tempo_real_rto:.0f} min) - {'DENTRO do RTO' if rto_ok else 'EXCEDEU o RTO!'}")
        
        if hora_normal:
            normal = datetime.strptime(hora_normal, "%H:%M")
            tempo_total = (normal - falha).seconds / 60
            tempo_wrt_real = (normal - restauro).seconds / 60
            mtd_ok = tempo_total <= mtd_min
            print(f"  Normal:    {hora_normal} (WRT real: {tempo_wrt_real:.0f} min, total: {tempo_total:.0f} min)")
            print(f"  MTD:       {'DENTRO do MTD' if mtd_ok else 'EXCEDEU o MTD!'}")
        
        print()
    
    print(f"{'='*70}")
    print()

In [ ]:
# Cenario 1: Incidente bem gerido
simular_incidente(
    nome="Loja Online - Incidente bem gerido",
    hora_falha="14:00",
    rpo_min=15, rto_min=60, wrt_min=120, mtd_min=240,
    hora_detecao="14:05",
    hora_restauro="14:45",
    hora_normal="16:30"
)

In [ ]:
# Cenario 2: Incidente que excede os limites
simular_incidente(
    nome="Loja Online - Incidente critico",
    hora_falha="14:00",
    rpo_min=15, rto_min=60, wrt_min=120, mtd_min=240,
    hora_detecao="14:30",
    hora_restauro="16:00",
    hora_normal="19:00"
)

In [ ]:
# Experimenta: simula o teu proprio incidente
simular_incidente(
    nome="O meu cenario",
    hora_falha="09:00",
    rpo_min=60, rto_min=120, wrt_min=180, mtd_min=480,
    hora_detecao="09:15",
    hora_restauro="10:30",
    hora_normal="13:00"
)

---

## 6. Comparador de cenarios

Compara metricas BIA de varios sistemas lado a lado.

In [ ]:
def comparar_cenarios(cenarios):
    """
    Compara varios cenarios BIA.
    
    Args:
        cenarios: Lista de dicionarios com nome, rpo, rto, wrt, mtd (em minutos)
    """
    # Cabecalho
    print(f"{'Cenario':<25} {'RPO':>8} {'RTO':>8} {'WRT':>8} {'MTD':>8} {'RTO+WRT':>8} {'Margem':>8} {'Valido':>8}")
    print(f"{'─'*25} {'─'*8} {'─'*8} {'─'*8} {'─'*8} {'─'*8} {'─'*8} {'─'*8}")
    
    for c in cenarios:
        soma = c['rto'] + c['wrt']
        margem = c['mtd'] - soma
        valido = "SIM" if soma <= c['mtd'] else "NAO"
        
        def fmt(mins):
            if mins < 60:
                return f"{mins}m"
            elif mins % 60 == 0:
                return f"{mins//60}h"
            else:
                return f"{mins//60}h{mins%60}m"
        
        print(f"{c['nome']:<25} {fmt(c['rpo']):>8} {fmt(c['rto']):>8} {fmt(c['wrt']):>8} {fmt(c['mtd']):>8} {fmt(soma):>8} {fmt(margem):>8} {valido:>8}")
    
    print()
    
    # Ranking por criticidade (menor MTD = mais critico)
    ranking = sorted(cenarios, key=lambda x: x['mtd'])
    print("Ranking por criticidade (mais critico primeiro):")
    for i, c in enumerate(ranking, 1):
        print(f"  {i}. {c['nome']} (MTD: {c['mtd']} min)")

In [ ]:
# Comparar todos os exemplos da aula
cenarios = [
    {"nome": "Loja Online",        "rpo": 15,   "rto": 60,  "wrt": 120, "mtd": 240},
    {"nome": "Hospital",           "rpo": 1,    "rto": 15,  "wrt": 30,  "mtd": 60},
    {"nome": "Universidade",       "rpo": 60,   "rto": 120, "wrt": 240, "mtd": 480},
    {"nome": "Banco",              "rpo": 0,    "rto": 30,  "wrt": 60,  "mtd": 120},
    {"nome": "Fabrica (SCADA)",    "rpo": 30,   "rto": 45,  "wrt": 120, "mtd": 240},
    {"nome": "Advogados",          "rpo": 240,  "rto": 240, "wrt": 180, "mtd": 480},
    {"nome": "Startup SaaS",       "rpo": 60,   "rto": 20,  "wrt": 40,  "mtd": 90},
    {"nome": "Autarquia",          "rpo": 1440, "rto": 480, "wrt": 240, "mtd": 1440},
    {"nome": "Logistica",          "rpo": 15,   "rto": 60,  "wrt": 60,  "mtd": 180},
    {"nome": "Escola",             "rpo": 480,  "rto": 240, "wrt": 120, "mtd": 1440},
]

comparar_cenarios(cenarios)

---

## 7. Exercicio: Calcula o RPO adequado para a frequencia de backup

Se o RPO e de X minutos, qual deve ser a frequencia dos backups?

In [ ]:
def analisar_backup(rpo_min, freq_backup_min):
    """
    Verifica se a frequencia de backup e adequada ao RPO.
    """
    adequado = freq_backup_min <= rpo_min
    
    print(f"RPO definido:             {rpo_min} minutos")
    print(f"Frequencia de backup:     a cada {freq_backup_min} minutos")
    print(f"Perda maxima possivel:    {freq_backup_min} minutos de dados")
    print()
    
    if adequado:
        print(f"RESULTADO: ADEQUADO")
        print(f"A frequencia de backup ({freq_backup_min} min) cumpre o RPO ({rpo_min} min).")
    else:
        print(f"RESULTADO: INADEQUADO!")
        print(f"A frequencia de backup ({freq_backup_min} min) e superior ao RPO ({rpo_min} min).")
        print(f"Na pior hipotese, podes perder {freq_backup_min} min de dados, mas so aceitas {rpo_min} min.")
        print(f"RECOMENDACAO: Fazer backups a cada {rpo_min} minutos ou menos.")
    print()

In [ ]:
# Clinica veterinaria: RPO de 4 horas mas backup diario (24h)
print("--- Clinica Veterinaria ---")
analisar_backup(rpo_min=240, freq_backup_min=1440)

# E-commerce: RPO de 15 minutos, backup a cada 15 minutos
print("--- E-commerce ---")
analisar_backup(rpo_min=15, freq_backup_min=15)

# Banco: RPO de 0, qualquer frequencia e insuficiente
print("--- Banco (RPO zero) ---")
analisar_backup(rpo_min=0, freq_backup_min=5)
print("Para RPO zero, e necessaria REPLICACAO SINCRONA, nao apenas backups.")

---

## 8. Desafio final

Usa as funcoes acima para analisar o seguinte cenario completo:

**Uma clinica medica privada tem:**
- Sistema de marcacoes online
- Sistema de registos clinicos eletronicos
- Sistema de faturacao
- Email e comunicacoes

**Tarefa:** Para cada sistema, define metricas BIA, valida-as e compara.

In [ ]:
# Resolve aqui o desafio!
# Dica: Usa as funcoes validar_bia() e comparar_cenarios()

# Exemplo para comecar:
# validar_bia("Marcacoes Online", rpo_min=?, rto_min=?, wrt_min=?, mtd_min=?)
# validar_bia("Registos Clinicos", rpo_min=?, rto_min=?, wrt_min=?, mtd_min=?)
# ...

# Depois compara:
# cenarios_clinica = [
#     {"nome": "Marcacoes",     "rpo": ?, "rto": ?, "wrt": ?, "mtd": ?},
#     {"nome": "Registos",      "rpo": ?, "rto": ?, "wrt": ?, "mtd": ?},
#     {"nome": "Faturacao",     "rpo": ?, "rto": ?, "wrt": ?, "mtd": ?},
#     {"nome": "Email",         "rpo": ?, "rto": ?, "wrt": ?, "mtd": ?},
# ]
# comparar_cenarios(cenarios_clinica)

---

## Resumo

| Metrica | Pergunta-chave |
|---------|---------------|
| **RPO** | Quanto dado posso perder? |
| **RTO** | Em quanto tempo quero recuperar? |
| **WRT** | Quanto tempo preciso para normalizar? |
| **MTD** | Maximo total de paragem toleravel? |
| **MOR** | Minimo para continuar a operar? |
| **MTBF** | De quanto em quanto tempo falha? |
| **MTTR** | Quanto tempo demora a reparar? |

**Regra de ouro:** `RTO + WRT <= MTD`

---

*Material de apoio para a UC TISI - MCIF - IPLeiria*